# Word Embeddings

Dans cette démonstration, nous allons voir comment utiliser des word embeddings pré-appris avec [`spacy`](https://spacy.io/) et [`gensim`](https://radimrehurek.com/gensim/).

Nous verrons ensuite comment apprendre nos propres embeddings.

In [ ]:
import itertools
import os
import pathlib
import sys
import re
import typing

import gensim.models
import gensim.models.phrases
import spacy
import warnings
warnings.filterwarnings("ignore")

## Modèle pré-entraîné de Spacy

Nous allons travailler avec un petit modèle pour l'anglais, qui contient des embeddings pour 20 000 mots.

In [ ]:
!python -m spacy download en_core_web_sm

## Utilisation

In [ ]:
nlp = spacy.load('en_core_web_sm')

doc = nlp("This is some text that I am processing with Spacy")

print(doc[3].vector)

print("Vecteur moyen")
print(doc.vector)

## Modèle pré-entraîné de Gensim

Gensim ne fournit pas d'embeddings pré-entraînés, mais un mécanisme pour en charger.

Quelques sources d'embeddings pré-entraînés :

- [Documentation word2vec de Google News](https://code.google.com/archive/p/word2vec/) ; [GoogleNews-vectors-negative300.bin.gz](https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing)

- [Dépôt de word embeddings du NLPL](http://vectors.nlpl.eu/repository/)

- [fasttext](https://fasttext.cc/docs/en/crawl-vectors.html)

- [Embeddings français de Jean-Philippe Fauconnier](https://fauconnier.github.io/)

In [ ]:
!wget https://embeddings.net/embeddings/frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin

In [ ]:
!ls -l 
model_file = "frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin"

## Utilisation de [`gensim.models.KeyedVectors`](https://radimrehurek.com/gensim/models/keyedvectors.html)

In [ ]:
model = gensim.models.KeyedVectors.load_word2vec_format(model_file, binary=True)
print("Taille du vocabulaire : ",len(model.vocab))

In [ ]:
for i, word in enumerate(model.vocab):
  print(word)
  if i > 50:
    break

In [ ]:
# Récupère le vecteur d'un mot
vector = model['facile']

# Le prétraitement doit être le même que pendant l'entraînement
words = "Ceci est une phrase transformée à l'aide de Gensim".lower().split(' ')
vectors = [(word, model[word]) for word in words if word in model.vocab]
print("Mots transformés :")
for word, vector in vectors:
  print(word)

In [ ]:
model.most_similar('berlin')

In [ ]:
print(model.similarity('berlin', 'munich'))
print(model.similarity('berlin', 'paris'))

## Opérations dans l'espace sémantique

À l'aide de [`gensim.models.keyedvectors.KeyedVectors.similar_by_vector`](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.similar_by_vector), nous allons définir une fonction d'analogie :

`x` est à `y` ce que `z` est à ?

*Ex.* : *Roi* est à *Reine* ce que *Oncle* est à ?

Réponse : *Tante*

In [ ]:
def semantic_analogy(x, y, z):
  return model.similar_by_vector(model[y] - model[x] + model[z])

In [ ]:
semantic_analogy("homme", "femme", "chien")

## Apprentissage de word embeddings

### Le corpus 20 Newsgroups

20 000 « posts » répartis en 20 thématiques.

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
!tar xzf 20news-bydate.tar.gz
!ls 20news-bydate-train/

In [ ]:
!ls 20news-bydate-test/

In [ ]:
contents = [p.read_text(encoding="latin-1")
            for p in pathlib.Path(".").glob("20news-bydate-*/*/*")]


print(f"{len(contents)} textes récupérés")

In [ ]:
print(contents[1])

In [ ]:
# Nettoyage
def preprocess(content: str) -> typing.List[str]:
  offset = content.find('\n\n')
  if i > 0:
    content = content[offset + 2:]
  sentences = (re.sub(r'[\!"#$%&\*+,-./:;<=>?@^_`()|~=]', " ", line).split()
               for line in content.splitlines()
               if not line.endswith("writes:"))
  return list(itertools.chain.from_iterable(sentences))


texts = [preprocess(content) for content in contents]

In [ ]:
print(texts[0])

## Détection de groupes de mots avec le Phraser de Gensim

In [ ]:
common_terms = ["of", "with", "without", "and", "or", "the", "a"]

phrases = gensim.models.phrases.Phrases(texts,
                                        common_terms=common_terms)

phraser = gensim.models.phrases.Phraser(phrases)
phrased_texts = list(phraser[texts])
len(phrased_texts)

In [ ]:
model = gensim.models.Word2Vec(
    phrased_texts, 
    min_count=3,
    size=200,
    workers=2,
    window=5,
    iter=30)

In [ ]:
model.most_similar('New_York')

In [ ]:
semantic_analogy("wind", "fly", "water")